In [3]:
import csv
import json
import re
import time
import urllib.error
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET
from pathlib import Path

import pandas as pd

CATEGORIES = ["cs.CL", "cs.LG", "cs.CV", "cs.AI", "cs.NE", "stat.ML"]
PAPERS_PER_CATEGORY = 1000
PAGE_SIZE = 100
SLEEP_SECONDS = 3.1

DATA_DIR = Path("data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = DATA_DIR / "arxiv_1000_per_category.csv"
PROGRESS_PATH = DATA_DIR / "arxiv_1000_per_category_progress.json"

ATOM = "{http://www.w3.org/2005/Atom}"
ARXIV = "{http://arxiv.org/schemas/atom}"
OPENSEARCH = "{http://a9.com/-/spec/opensearch/1.1/}"
API_URL = "https://export.arxiv.org/api/query"
COLUMNS = ["paper_id", "title", "abstract", "category", "published", "updated"]


def clean(text):
    return re.sub(r"\s+", " ", text or "").strip()


def arxiv_id(value):
    value = clean(value)
    value = value.removeprefix("https://arxiv.org/abs/").removeprefix("http://arxiv.org/abs/")
    return re.sub(r"v\d+$", "", value)


def load_existing_csv():
    if not CSV_PATH.exists():
        return pd.DataFrame(columns=COLUMNS)
    df = pd.read_csv(CSV_PATH)
    df = df.drop_duplicates(subset="paper_id", keep="first")
    df = df[df["category"].isin(CATEGORIES)].copy()
    df.to_csv(CSV_PATH, index=False)
    return df


def load_progress():
    if not PROGRESS_PATH.exists():
        return {}
    return json.loads(PROGRESS_PATH.read_text(encoding="utf-8"))


def save_progress(progress):
    PROGRESS_PATH.write_text(json.dumps(progress, indent=2), encoding="utf-8")


def append_rows(rows):
    if not rows:
        return
    file_exists = CSV_PATH.exists()
    with CSV_PATH.open("a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=COLUMNS)
        if not file_exists:
            writer.writeheader()
        writer.writerows(rows)


def fetch_category_page(category, start, retries=4):
    params = {
        "search_query": f"cat:{category}",
        "start": start,
        "max_results": PAGE_SIZE,
        "sortBy": "submittedDate",
        "sortOrder": "descending",
    }
    url = API_URL + "?" + urllib.parse.urlencode(params)

    for attempt in range(1, retries + 1):
        try:
            request = urllib.request.Request(
                url,
                headers={"User-Agent": "final-nlp-project/1.0 (mailto:student@example.com)"},
            )
            with urllib.request.urlopen(request, timeout=60) as response:
                root = ET.fromstring(response.read())
            total_text = root.findtext(f"{OPENSEARCH}totalResults", "")
            total_results = int(total_text) if total_text.isdigit() else None
            return root, total_results
        except (urllib.error.URLError, TimeoutError, ET.ParseError) as exc:
            wait = max(SLEEP_SECONDS, 5 * attempt)
            print(f"    fetch failed attempt {attempt}/{retries}: {exc}; waiting {wait:.1f}s")
            time.sleep(wait)
    raise RuntimeError(f"Could not fetch {category} page starting at {start}")


def parse_entry(entry):
    primary = entry.find(f"{ARXIV}primary_category")
    primary_category = primary.attrib.get("term", "") if primary is not None else ""
    return {
        "paper_id": arxiv_id(entry.findtext(f"{ATOM}id", "")),
        "title": clean(entry.findtext(f"{ATOM}title", "")),
        "abstract": clean(entry.findtext(f"{ATOM}summary", "")),
        "category": primary_category,
        "published": entry.findtext(f"{ATOM}published", ""),
        "updated": entry.findtext(f"{ATOM}updated", ""),
    }


existing_df = load_existing_csv()
progress = load_progress()
seen_ids = set(existing_df["paper_id"].astype(str)) if len(existing_df) else set()

print(f"Resume CSV: {CSV_PATH}")
print(f"Progress file: {PROGRESS_PATH}")
print(f"Already saved rows: {len(existing_df)}")
if len(existing_df):
    print("Existing category counts:")
    print(existing_df["category"].value_counts().sort_index())
print()

for category in CATEGORIES:
    current_count = int((existing_df["category"] == category).sum()) if len(existing_df) else 0
    start = int(progress.get(category, {}).get("next_start", 0))

    print("=" * 80)
    print(f"Category {category}: have {current_count}/{PAPERS_PER_CATEGORY}; resume start={start}")

    if current_count >= PAPERS_PER_CATEGORY:
        print(f"Category {category}: complete, skipping")
        continue

    while current_count < PAPERS_PER_CATEGORY:
        root, total_results = fetch_category_page(category, start)
        entries = root.findall(f"{ATOM}entry")
        if not entries:
            print(f"Category {category}: no entries returned at start={start}; stopping")
            break

        new_rows = []
        primary_matches = 0
        duplicates = 0

        for entry in entries:
            row = parse_entry(entry)
            if row["category"] == category:
                primary_matches += 1
            else:
                continue

            if row["paper_id"] in seen_ids:
                duplicates += 1
                continue

            new_rows.append(row)
            seen_ids.add(row["paper_id"])
            current_count += 1
            if current_count >= PAPERS_PER_CATEGORY:
                break

        append_rows(new_rows)
        start += PAGE_SIZE
        progress[category] = {
            "next_start": start,
            "saved_count": current_count,
            "target_count": PAPERS_PER_CATEGORY,
            "last_updated": time.strftime("%Y-%m-%d %H:%M:%S"),
        }
        save_progress(progress)

        print(
            f"Category {category} page start={start - PAGE_SIZE}: "
            f"entries={len(entries)}, primary_matches={primary_matches}, "
            f"new_saved={len(new_rows)}, duplicates={duplicates}, "
            f"total_saved={current_count}/{PAPERS_PER_CATEGORY}, "
            f"api_totalResults={total_results}"
        )
        print(f"    CSV saved after this page: {CSV_PATH}")

        if total_results is not None and start >= total_results:
            print(f"Category {category}: reached API totalResults={total_results}; stopping")
            break

        if current_count < PAPERS_PER_CATEGORY:
            time.sleep(SLEEP_SECONDS)

print("=" * 80)
df = load_existing_csv()
print(f"Final saved rows: {len(df)}")
print("Final category counts:")
print(df["category"].value_counts().sort_index())
df


Resume CSV: data/raw/arxiv_1000_per_category.csv
Progress file: data/raw/arxiv_1000_per_category_progress.json
Already saved rows: 0

Category cs.CL: have 0/1000; resume start=0
Category cs.CL page start=0: entries=100, primary_matches=80, new_saved=80, duplicates=0, total_saved=80/1000, api_totalResults=109616
    CSV saved after this page: data/raw/arxiv_1000_per_category.csv
Category cs.CL page start=100: entries=100, primary_matches=70, new_saved=70, duplicates=0, total_saved=150/1000, api_totalResults=109616
    CSV saved after this page: data/raw/arxiv_1000_per_category.csv
Category cs.CL page start=200: entries=100, primary_matches=64, new_saved=64, duplicates=0, total_saved=214/1000, api_totalResults=109616
    CSV saved after this page: data/raw/arxiv_1000_per_category.csv
Category cs.CL page start=300: entries=100, primary_matches=57, new_saved=57, duplicates=0, total_saved=271/1000, api_totalResults=109616
    CSV saved after this page: data/raw/arxiv_1000_per_category.csv
C

,paper_id,title,abstract,category,published,updated
0,2605.21465,Leveraging LLMs for Grammar Adaptation: A Stud...,"In model-driven engineering, metamodel evoluti...",cs.CL,2026-05-20T17:51:51Z,2026-05-20T17:51:51Z
1,2605.21463,Mem-$π$: Adaptive Memory through Learning When...,"We present Mem-$π$, a framework for adaptive m...",cs.CL,2026-05-20T17:51:05Z,2026-05-20T17:51:05Z
2,2605.21403,Quantifying the cross-linguistic effects of sy...,"Agreement attraction errors, in which a verb e...",cs.CL,2026-05-20T17:02:17Z,2026-05-20T17:02:17Z
3,2605.21391,Post-Hoc Understanding of Metaphor Processing ...,Metaphor requires a language model to resolve ...,cs.CL,2026-05-20T16:45:59Z,2026-05-20T16:45:59Z
4,2605.21369,Findings of the Fifth Shared Task on Multiling...,This paper describes the fifth edition of the ...,cs.CL,2026-05-20T16:35:09Z,2026-05-20T16:35:09Z
...,...,...,...,...,...,...
5995,2601.17510,"""Rebuilding"" Statistics in the Age of AI: A To...","This article presents the full, original recor...",stat.ML,2026-01-24T16:15:04Z,2026-01-24T16:15:04Z
5996,2601.17374,Error Analysis of Bayesian Inverse Problems wi...,Data-driven methods for the solution of invers...,stat.ML,2026-01-24T08:45:27Z,2026-03-10T21:30:50Z
5997,2601.17160,Information-Theoretic Causal Bounds under Unme...,We develop a data-driven information-theoretic...,stat.ML,2026-01-23T20:47:48Z,2026-02-20T19:10:36Z
5998,2601.16597,Efficient Learning of Stationary Diffusions wi...,Learning a stationary diffusion amounts to est...,stat.ML,2026-01-23T10:00:06Z,2026-01-29T13:58:58Z
